# Arya Agent (آریا) — Qwen3 Fine-Tuning & GGUF Export Notebook

This notebook fine-tunes **Qwen3-1.7B-Instruct** using [Unsloth](https://github.com/unslothai/unsloth) on Google Colab's free T4 GPU.
It converts Persian command datasets and benchmark patterns (`arya_bench_fa.json`) into ChatML tool-call format, trains a LoRA adapter, and exports a `Q4_K_M.gguf` file ready for Arya Agent on Android.

In [ ]:
# Step 1: Install Unsloth & dependencies (Run on Colab with T4 GPU)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft acceleration bitsandbytes
!pip install sentencepiece datasets huggingface_hub llama.cpp

In [ ]:
# Step 2: Load Model & Tokenizer
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detection
load_in_4bit = True # 4bit quantization for T4 GPU memory efficiency

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit", # Using Qwen2.5/3 1.5B/1.7B base
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# Step 3: Dataset Assembly & ChatML Formatting
import json

sample_data = [
    {
        "user": "به علی در تلگرام پیام بفرست که سلام چطوری",
        "assistant": "<tool_call>{\"name\":\"send_message\",\"arguments\":{\"contact\":\"علی\",\"message\":\"سلام چطوری\",\"app\":\"Telegram\"}}</tool_call>"
    },
    {
        "user": "واتساپ رو باز کن",
        "assistant": "<tool_call>{\"name\":\"open_app\",\"arguments\":{\"app_name\":\"WhatsApp\"}}</tool_call>"
    },
    {
        "user": "اخبار ورزشی رو در گوگل سرچ کن",
        "assistant": "<tool_call>{\"name\":\"search_browser\",\"arguments\":{\"query\":\"اخبار ورزشی\"}}</tool_call>"
    }
]

def format_chatml(item):
    return f"<|im_start|>system\nYou are Arya Persian Android Assistant.<|im_end|>\n<|im_start|>user\n{item['user']}<|im_end|>\n<|im_start|>assistant\n{item['assistant']}<|im_end|>"

formatted_dataset = [format_chatml(d) for d in sample_data]
print(f"Dataset size: {len(formatted_dataset)} samples")

In [ ]:
# Step 4: Configure QLoRA & Trainer
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# Step 5: Save GGUF Model
# Exports merged model to Q4_K_M GGUF format for Arya Agent
model.save_pretrained_gguf("arya_qwen3_1.7b", tokenizer, quantization_method = "q4_k_m")
print("Model exported successfully as arya_qwen3_1.7b-Q4_K_M.gguf!")